# Silver: prepaid services

**Audience:** data engineers validating medallion architecture and AIDP lineage.

**Prerequisites:** the canonical lab assets, shared compute and five job parameters.

**Learning goals:** trace governed transformations, verify isolation, and inspect deterministic results.


In [ ]:
import re
from functools import reduce
from pyspark.sql import Window, functions as F

# oidlUtils is injected by AIDP Workbench; no import is required.
def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

participant_match = re.fullmatch(r"u([1-9][0-9]*)", participant_key)
if participant_match is None or int(participant_match.group(1)) < 101:
    raise ValueError("Invalid participant_key")
if lab_id != "telco_lineage":
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

layer_prefixes = {"landing": "01_landing", "bronze": "02_bronze", "silver": "03_silver", "gold": "04_gold"}

def table(layer, logical_name):
    return f"aidp_lab.oci_{layer}.{participant_key}_{lab_id}_{logical_name}"

def location(layer, logical_name):
    return f"oci://{bucket_name}@{objectstorage_namespace}/{layer_prefixes[layer]}/users/{participant_key}/{lab_id}/{logical_name}/"

def write_delta(frame, layer, logical_name, _ddl):
    target = table(layer, logical_name)
    (frame.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target))
    actual = spark.table(target).count()
    assert actual == frame.count(), f"Delta count mismatch for {logical_name}"
    print(f"Delta {layer}.{logical_name}: {actual} rows")


## Transformation

Run this cell once. It is idempotent and checks its row-level contract.


In [ ]:
customers = spark.table(table("silver", "customer_master")).select("customer_id").withColumn("_customer_ok", F.lit(True))
products = (spark.table(table("silver", "product_catalog")).filter(F.col("service_type") == "PREPAID")
    .select("product_id", "product_name", "product_family").withColumn("_product_ok", F.lit(True)))
lines = spark.table(table("bronze", "prepaid_lines"))
checked_lines = lines.join(customers, "customer_id", "left").join(products, "product_id", "left")
line_reason = (F.when(F.col("_customer_ok").isNull(), F.lit("orphan_customer"))
    .when(F.col("_product_ok").isNull(), F.lit("invalid_product")))
checked_lines = checked_lines.withColumn("_reason", line_reason)
line_issues = (checked_lines.filter(F.col("_reason").isNotNull())
    .select(F.lit(participant_key).alias("participant_key"), F.lit("prepaid_lines").alias("dataset"),
        "source_row_id", F.col("line_id").alias("record_key"), F.col("_reason").alias("reason_code"),
        F.current_timestamp().alias("quarantined_at")))
valid_lines = checked_lines.filter(F.col("_reason").isNull())

recharges = (spark.table(table("bronze", "prepaid_recharges"))
    .withColumn("amount_value", F.col("amount").cast("decimal(12,2)")))
checked_recharges = recharges.join(valid_lines.select("line_id").withColumn("_line_ok", F.lit(True)), "line_id", "left")
recharge_reason = (F.when(F.col("_line_ok").isNull(), F.lit("orphan_line"))
    .when(F.col("amount_value") <= 0, F.lit("negative_amount")))
checked_recharges = checked_recharges.withColumn("_reason", recharge_reason)
recharge_issues = (checked_recharges.filter(F.col("_reason").isNotNull())
    .select(F.lit(participant_key).alias("participant_key"), F.lit("prepaid_recharges").alias("dataset"),
        "source_row_id", F.col("recharge_id").alias("record_key"), F.col("_reason").alias("reason_code"),
        F.current_timestamp().alias("quarantined_at")))
recharge_value = (checked_recharges.filter(F.col("_reason").isNull()).groupBy("line_id")
    .agg(F.sum("amount_value").alias("monthly_value")))
prepaid_service = (valid_lines.join(recharge_value, "line_id", "left")
    .select("participant_key", F.col("line_id").alias("service_id"), F.col("msisdn").alias("service_number"),
        F.lit("PREPAID").alias("service_type"), "customer_id", "product_id",
        F.lower("status").alias("status"), F.coalesce("monthly_value", F.lit(0)).cast("decimal(14,2)").alias("monthly_value")))
write_delta(prepaid_service, "silver", "prepaid_service", "participant_key STRING, service_id STRING, service_number STRING, service_type STRING, customer_id STRING, product_id STRING, status STRING, monthly_value DECIMAL(14,2)")
line_issues.unionByName(recharge_issues).write.format("delta").mode("overwrite").save(location("silver", "_quality/prepaid"))
assert prepaid_service.count() == 645


## Exercise and common pitfall

**Exercise:** follow one customer or service identifier into the next task and explain every derived column.

**Answer scaffold:** identify the source table, join key, transformation and target column.

**Pitfall:** never replace the job parameters with participant-specific literals; doing so breaks canonical hashes and isolation.

**Extension:** inspect the resulting entity and column lineage in Master Catalog.
